# NB7 — gradient spectrum at M_A (go / no-go for the SVD arms)

Checks the add-on on this image, then for every seed with a finished Stage A: the skill-B gradient at M_A for each adapted layer, how concentrated it is, how much of it falls inside A's protected subspace, the predicted first-step speed-up of SVD over Kaiming initialization (SGD and AdamW), the SVD bases the SVD arms start from, and (tier 3) a larger protection basis with up to 256 directions. The work is balanced over both GPUs while the tests run on the CPU: about 35–40 minutes for three seeds.

**Accelerator:** GPU T4 x2 · **Internet:** OFF · **Attach:** pcl-next code, model, NB2, all NB3 outputs

Run with **Save Version → Save & Run All (Commit)**. Afterwards: read `spectrum/spectrum_report.md`. Its last line says whether NB8 will run the SVD-initialized arms. NB8 needs this output.

In [ ]:
SESSION_HOURS = 12   # your GPU session limit (see the notebook settings panel)
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})
# --- Follow-up add-on (pclx lives in notebooks_2/ of the same code dataset) ------------
NEXT = WORK / 'notebooks_2'
assert (NEXT / 'pclx' / 'cli.py').is_file(), 'attach the pcl-next code dataset (it contains notebooks_2/)'
sys.path.insert(0, str(NEXT))
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in (str(WORK), str(NEXT), os.environ.get('PYTHONPATH', '')) if p)
from pclx import plan

def pclx(*args):
    print('$ pclx', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pclx', *map(str, args)], check=True, cwd=WORK)

def pclx_on_gpus(commands, log_dir):
    """One pclx command per T4 (at most two), logs streamed live."""
    commands = [json.dumps([sys.executable, '-m', 'pclx', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

found_next = plan.discover_next([INPUT, WORKING], exclude=code_roots[0])
print({'stage_a_seeds': plan.stage_a_seeds(found_next), 'spectra': sorted(found_next['spectra']),
       'finished_items': sorted(k for k in found_next['items'] if plan.item_complete(found_next, *k))})

In [ ]:
MODEL = kaggle.one(found, 'models', 'pinned model Dataset (made from the NB1 output)')
M0_CONFIG = kaggle.m0_config(MODEL, WORKING / 'configs' / 'm0.yaml')
DATA = kaggle.one(found, 'data', 'NB2 output (data)')
PILOT = kaggle.one(found, 'pilot', 'NB2 output (pilot)')
SETTINGS = kaggle.study_settings(PILOT, WORKING / 'study.yaml')
settings = yaml.safe_load(SETTINGS.read_text())
assert settings['precision_validated'], 'The pilot did not validate FP16. Read pilot/pilot_report.md first.'
print(settings)

In [ ]:
BUILD_K256 = True   # the tier-3 arm protected_k256 needs it (~15 min per seed); False skips that arm
# The add-on's tests (unit tests + the follow-up round in miniature, ~3 min of CPU) run while the GPUs work.
tests = subprocess.Popen([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '--basetemp', '/tmp/pytest-next',
                          str(NEXT / 'tests')], cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tasks = plan.nb7_tasks(found_next, build_k256=BUILD_K256)
queues = plan.balance(tasks)   # longest task first onto the less-loaded GPU
print('NB7 tasks per GPU:', queues)
commands = [['spectrum', '--search', INPUT, WORKING, '--exclude', code_roots[0], '--tasks', *queue,
             '--model-config', M0_CONFIG, '--data-root', DATA, '--out', WORKING / 'spectrum'] for queue in queues]
if commands:
    pclx_on_gpus(commands, WORKING / 'logs')
log = tests.communicate()[0]
print(log[-2000:])
if tests.returncode != 0:
    for marker in [*WORKING.rglob('spectrum_summary.json'), *WORKING.rglob('k256_summary.json')]:
        marker.unlink()   # outputs from an image that fails the tests must not be used by NB8
    raise AssertionError('the follow-up add-on failed its tests on this image; outputs were invalidated')
pclx('spectrum-report', '--search', INPUT, WORKING, '--exclude', code_roots[0], '--out', WORKING / 'spectrum')
show(WORKING / 'spectrum' / 'spectrum_report.md')